# Access PACE data with OPeNDAP

In [ ]:
import xarray as xr
import datetime as dt
import earthaccess
import matplotlib.pyplot as plt
import numpy as np
# import pydap-specific tools
from pydap.client import get_cmr_urls, open_url
from pydap.client import to_netcdf as dap_to_netcdf

## Search for OPeNDAP URLs

Will need the concept collection ID (ccid), or the DOI of the data of interest at a minimum.

Can pass additional filters to narrow down the search, such as a time range, bounding box, etc.


In [ ]:
PACE_ccid = "C3620140256-OB_CLOUD" # version 3.1 
time_range=[dt.datetime(2025, 1, 1), dt.datetime(2025, 3, 30)]
cmr_urls = get_cmr_urls(ccid=PACE_ccid, limit=1000, time_range=time_range) # limit by default = 50

print("################################################ \n We found a total of ", len(cmr_urls), "OPeNDAP URLS!!!\n################################################")

In [ ]:
cmr_urls[:5]

## Identify only those granules associated with 4km resolution

The CMR returns URLs that belong to different resolutions. We need to further filter these to that ALL urls point to a homogeneous resource.


In [ ]:
chlor_a_urls = [url for url in cmr_urls if "DAY.CHL.V3_1.chlor_a.4km" in url]
len(chlor_a_urls)

In [ ]:
chlor_a_urls[:5]

## EDL Authentication

This is a necessary step to download any data, which includes metadata, from a remote file.

There are many ways to authenticate with OPeNDAP in Python, below we use earthaccess as it can abstract the authentication to use tokens to retrieve data.

In [ ]:
auth = earthaccess.login(strategy="netrc", persist=True) # you will be promted to add your EDL credentials

# pass Token Authorization to a new Session.
my_session = session=auth.get_session()

## Subset remote data 

Reading metadata from one granule within a Collection is important because it allows one to **identify variables of interest to subset by variable name**.

OPeNDAP supports subsetting by variable names and by spatial subsets via the use of **Constraint Expressions**. PyDAP can construct DAP4 constraint expressions when the user provides a list of variable names and the dimension slices. As of now, identifying the spatial subset requires download coordinate data from remote granules. 


###  Subset by Variable name: How to provide list of names for subset using the DAP4 protocol?

Many of NASA files are hierarchical, meaning that there can be one of more Groups inside the file, and DAP4 fully supports it. 

What is a Group? A Group acts like a folder inside a computer filesystem, and to correctly identifying a variable by its name, one neeed to specify both path and name of the variable. This helps avoid **name collisions**. A name that integrates the full path to the variable is the `Fully Qualifying Name`.


### Subset by coordinate values

Consider the case of an area of interest defined by bounding box with the following values (in python):
```python
# Min/max of lon values
minLon, maxLon = -96, 10

# Min/Max of lat values
minLat, maxLat = 6, 70
```

With DAP4 protocol, one needs to download data and identify the indexes of the array. In python this is easy and is demonstrated below. 


In [ ]:

# create an Xarray Dataset object. It eagerly downloads all dimension data, which in this case
# it facilitates our workflow since `latitude` & `longitude` are dimension data.
ds = xr.open_dataset(chlor_a_urls[0].replace("https","dap4"), session=my_session, engine='pydap')


# Min/max of lon values
minLon, maxLon = -96, 10
# Min/Max of lat values
minLat, maxLat = 6, 70

lat, lon = ds['lat'].values, ds['lon'].values

# Identify ALL lat/lon data inside the bounding box
iLon = np.where((lon>minLon)&(lon < maxLon))[0] # 1D array
iLat= np.where((lat>minLat)&(lat < maxLat))[0] # 1D array


### Define parameters to pass to pydap to stream data from remote OPeNDAP servers

These are:

1. List of OPeNDAP URLs.
2. `output_path`: Where the data will be stored.
3. `keep_variables`: A list of variables of interest. These will be downloaded. The variable names must be fully qualifying names.
4. `dim_slices`: This is a dictionary where the key is the dimension name (fully qualifying) and the values is a Tuple of first and last index. These together define the `slice` that will be applied to all relevant variables.
5. `session`: A (requests) session object that contains the EDL authentication information. This is the `my_session` object above. 


In [ ]:
output_path = "data/"

keep_variables = ['/lon', '/lat', "/chlor_a"]
dim_slices = {'/lat': (iLat[0], iLat[-1]), '/lon': (iLon[0], iLon[-1])}


## Stream data

In [ ]:
%%time
dap_to_netcdf(chlor_a_urls, session=my_session, keep_variables=keep_variables, dim_slices=dim_slices, output_path=output_path)

## Inspect local files

In [ ]:
nds = xr.open_mfdataset(output_path+"PACE_OCI.*.nc4", concat_dim='time', parallel=True, combine="nested")
nds